# Red-Green-Refactor with pytest: -k, -x, and assertion introspection

This notebook walks through the red-green-refactor cycle using pytest's built-in flags. The goal is to practice the TDD loop: write a failing test (red), make it pass (green), then clean up (refactor) — and use `-k`, `-x`, and assertion introspection to stay focused.

## Setup

We need pytest installed. No external dependencies for this notebook.

In [ ]:
# Install pytest if not already available
!pip install pytest -q

## Step 1: Red — write a failing test

We start with a function that doesn't exist yet. The test should fail because the implementation is missing.

In [ ]:
%%writefile test_calculator.py
def add(a, b):
    """Add two numbers."""
    return a + b


def test_add_positive():
    assert add(1, 2) == 3


def test_add_negative():
    assert add(-1, -2) == -3


def test_add_zero():
    assert add(0, 5) == 5

In [ ]:
!python -m pytest test_calculator.py -v

All three tests pass — that's the green state. Now let's add a test that will fail, pushing us back to red.

In [ ]:
%%writefile test_calculator.py
def add(a, b):
    """Add two numbers."""
    return a + b


def subtract(a, b):
    """Subtract b from a."""
    raise NotImplementedError("subtract not implemented yet")


def test_add_positive():
    assert add(1, 2) == 3


def test_add_negative():
    assert add(-1, -2) == -3


def test_subtract_basic():
    assert subtract(5, 3) == 2


def test_subtract_negative():
    assert subtract(-1, -2) == 1

In [ ]:
!python -m pytest test_calculator.py -v

The `subtract` tests fail — that's red. Now we make them pass.

In [ ]:
%%writefile test_calculator.py
def add(a, b):
    """Add two numbers."""
    return a + b


def subtract(a, b):
    """Subtract b from a."""
    return a - b


def test_add_positive():
    assert add(1, 2) == 3


def test_add_negative():
    assert add(-1, -2) == -3


def test_subtract_basic():
    assert subtract(5, 3) == 2


def test_subtract_negative():
    assert subtract(-1, -2) == 1

In [ ]:
!python -m pytest test_calculator.py -v

All green. Now refactor: extract a common pattern if one exists, clean up naming.

## Step 2: Using `-k` to select tests by expression

The `-k` flag filters tests by name expression. It matches against the test function name and supports boolean operators.

In [ ]:
# Run only tests with 'add' in the name
!python -m pytest test_calculator.py -k 'add' -v

In [ ]:
# Run only tests with 'negative' in the name
!python -m pytest test_calculator.py -k 'negative' -v

In [ ]:
# Exclude tests with 'subtract' using not
!python -m pytest test_calculator.py -k 'not subtract' -v

During the red-green-refactor loop, `-k` lets you focus on just the tests related to the feature you're working on without running the full suite.

## Step 3: Using `-x` to stop on first failure

The `-x` flag stops the test run after the first failure. This is useful during TDD because you want to see the failure immediately, fix it, and re-run — rather than scrolling past 50 passing tests to find the one that broke.

In [ ]:
# Intentionally break a test to see -x in action
%%writefile test_calculator.py
def add(a, b):
    """Add two numbers."""
    return a + b


def subtract(a, b):
    """Subtract b from a."""
    return a - b


def test_add_positive():
    assert add(1, 2) == 3


def test_add_negative():
    assert add(-1, -2) == -3


def test_subtract_basic():
    assert subtract(5, 3) == 99  # intentionally wrong


def test_subtract_negative():
    assert subtract(-1, -2) == 1

In [ ]:
# Without -x: runs all tests, reports failures at the end
!python -m pytest test_calculator.py -v 2>&1 | head -20

In [ ]:
# With -x: stops at the first failure
!python -m pytest test_calculator.py -x -v 2>&1

Notice how `-x` stops after `test_subtract_basic` fails — `test_subtract_negative` never runs. This keeps the feedback loop tight during TDD.

## Step 4: Assertion introspection

pytest rewrites `assert` statements at import time to produce detailed failure output. When an assertion fails, pytest shows the exact values of both sides of the comparison.

In [ ]:
%%writefile test_introspection.py
def get_user():
    return {"name": "Alice", "age": 30, "roles": ["admin", "editor"]}


def test_user_name():
    user = get_user()
    assert user["name"] == "Bob"  # wrong — will show the actual value


def test_user_age():
    user = get_user()
    assert user["age"] > 25  # passes


def test_user_roles():
    user = get_user()
    assert "viewer" in user["roles"]  # wrong — will show the actual list


def test_user_dict():
    user = get_user()
    expected = {"name": "Alice", "age": 31, "roles": ["admin"]}  # age wrong, roles incomplete
    assert user == expected

In [ ]:
!python -m pytest test_introspection.py -v 2>&1

Each failure shows the exact values pytest found. For the dict comparison, it shows a diff of which keys differ. This is assertion introspection — pytest's rewriting of `assert` to give you diagnostic output without adding any extra code.

Compare this to `unittest` where `self.assertEqual` gives similar output but requires method calls. With pytest, plain `assert` is enough.

## Putting it together: the TDD loop

The full cycle:

1. **Red**: Write a test that describes the behavior you want. Run it — it should fail.
2. **Green**: Write the minimum code to make the test pass. Run it — all tests should pass.
3. **Refactor**: Clean up the code while keeping tests green. Run the full suite to confirm nothing broke.

Use `-k` to focus on the tests for the current feature, `-x` to stop at the first failure during iteration, and assertion introspection to understand exactly what went wrong without adding print statements.

In [ ]:
# Clean up
!rm -f test_calculator.py test_introspection.py